In [25]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization

from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives import hashes

In [26]:
def generate_keys():
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048
    )

    public_key = private_key.public_key()

    # збереження приватного ключа
    with open("private.pem", "wb") as f:
        f.write(private_key.private_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption()
        ))

    # збереження публічного ключа
    with open("public.pem", "wb") as f:
        f.write(public_key.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo
        ))

    print("Ключі згенеровано")

In [27]:
def load_public_key():
    with open("public.pem", "rb") as f:
        return serialization.load_pem_public_key(f.read())

def load_private_key():
    with open("private.pem", "rb") as f:
        return serialization.load_pem_private_key(f.read(), password=None)

In [28]:
def encrypt_file(input_file, output_file):
    public_key = load_public_key()

    chunk_size = 190  # для RSA 2048 + OAEP

    with open(input_file, "rb") as f_in, open(output_file, "wb") as f_out:
        while chunk := f_in.read(chunk_size):
            encrypted = public_key.encrypt(
                chunk,
                padding.OAEP(
                    mgf=padding.MGF1(algorithm=hashes.SHA256()),
                    algorithm=hashes.SHA256(),
                    label=None
                )
            )

            # записуємо довжину + дані
            f_out.write(len(encrypted).to_bytes(4, 'big'))
            f_out.write(encrypted)

In [29]:
def decrypt_file(input_file, output_file):
    private_key = load_private_key()

    with open(input_file, "rb") as f_in, open(output_file, "wb") as f_out:
        while True:
            length_bytes = f_in.read(4)
            if not length_bytes:
                break

            length = int.from_bytes(length_bytes, 'big')
            encrypted_chunk = f_in.read(length)

            decrypted = private_key.decrypt(
                encrypted_chunk,
                padding.OAEP(
                    mgf=padding.MGF1(algorithm=hashes.SHA256()),
                    algorithm=hashes.SHA256(),
                    label=None
                )
            )

            f_out.write(decrypted)

In [30]:
generate_keys()
encrypt_file("hollyshit.txt", "encrypted.bin")
decrypt_file("encrypted.bin", "decrypted.txt")

Ключі згенеровано
